# AA3
Implementem suas próprias versões dos métodos:
- regressão linear,
- regressão localmente ponderada (LWR)
- e regressão logística.

Dica:
- implemente um método fit,
  - para ajustar os parâmetros com base nos dados,
- e um método predict para realizar predições usando os parâmetros. 

Teste seus métodos usando o dataset exemplo disponível no Capítulo 2 do livro "GÉRON, A. Mãos à obra:
  - aprendizado de máquina com Scikit-Learn, Keras e TensorFlow.
  - Conceitos, ferramentas e técnicas para a construção de sistemas inteligentes

- Use as mesmas técnicas disponíveis na biblioteca sklearn nestes dados e compare os resultados obtidos com a sua implementação.

Como o principal objetivo desta atividade é aprofundar o seu entendimento a respeito deste tópico,
é fortemente recomendável que você não use qualquer tipo de assistente de código para completá-la.

In [ ]:
import numpy as np
import pandas as pd
import os

## Função de Custo

In [ ]:
def mse(y: np.ndarray, y_pred: np.ndarray) -> np.float64:
    m = np.size(y)
    return np.sum(np.square(y - y_pred) / (2.0*m))

def loss_fn(X: np.ndarray, y: np.ndarray, weigths: np.ndarray) -> np.float64:
    y_pred = X @ weigths
    return mse(y, y_pred)

loss_fn(np.array([[1, 1], [1, 1], [1, 1]]), np.array([2, 2, 2]), np.array([1, 1]))

## Gradiente

In [ ]:
def gradient_linear(X: np.ndarray, y: np.ndarray, weigths: np.ndarray) -> np.ndarray:
    m = np.size(y)

    y_pred = X @ weigths

    dp_l2 = X.T @ (y_pred - y) / m
    return dp_l2

gradient_linear(np.array([[1, 1], [1, 1], [1, 1]]), np.array([1, 1, 1]), np.array([1, 1]))

# Regressão Linear

In [ ]:
def fit_linear_regression(X_input: np.ndarray, y:np.ndarray, num_iter: int, lr: float):
    m = np.size(y)
    ones = np.ones((m, 1))
    X = np.hstack((ones, X_input))

    np.random.seed(42)
    weigths = np.random.rand(X.shape[1])

    print(f"First Error: {loss_fn(X, y, weigths):.10f}")

    for i in range(num_iter):
        weigths -= lr*gradient_linear(X, y, weigths)
    print(f"Last Error:  {loss_fn(X, y, weigths):.10f}")
    return weigths

X = np.array([[2], [2], [2]])
y = np.array([3, 3, 3])
ITER = 1000
LR = 0.01

fit_linear_regression(X, y, ITER, LR)

## Testing

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

csv_path = os.path.join(os.path.join("datasets", "housing"), "housing.csv")
df = pd.read_csv(csv_path, index_col=False)
df["ocean_proximity"] = 0.0
df = df.fillna(df.median())  # Fill NaN with column medians
df["rooms_per_house"] = df["total_rooms"] / df["households"]

TARGET_COL = 'median_house_value'
FEATURES_COL = ["housing_median_age", "rooms_per_house", "median_income"]

# Standardize X and y
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X = scaler_X.fit_transform(df[FEATURES_COL].to_numpy())
y = scaler_y.fit_transform(df[TARGET_COL].to_numpy().reshape(-1, 1)).flatten()

ITER = 10000
LR = 0.001

fit_linear_regression(X, y, ITER, LR)
# m = np.size(y)
# ones = np.ones((m, 1))
# X_test = np.hstack((ones, X))

# custom_preds = X_test @ weights

# print("Custom Result:", mse(y, custom_preds))

comparison_model = LinearRegression()
comparison_model.fit(X, y)

y_pred = comparison_model.predict(X)

print(f"Lib Result: {mse(y, y_pred):.10f}");

# LWR

In [ ]:
def lc_weight(X: np.ndarray, x0: np.ndarray, bandwidth: np.float64) -> np.ndarray:
    # print("original:", X)
    er = X - x0
    # print("subtracted:", er)
    er = np.square(er)
    # print("squared:", er)
    # obtaining the distance
    er = np.sum(er, axis=1)
    # print("summed:", er)
    den = 2.0*bandwidth*bandwidth
    er /= den
    # print("divided:", er)
    er = -er
    er = np.exp(er)
    # print("exp:", er)
    # turning from vector to matrix to simplify the calculations
    er = np.diag(er)
    # print("diag:", er)
    return er


In [ ]:
def lc_w_gradient(x0: np.ndarray, X: np.ndarray, y: np.ndarray, weigths: np.ndarray, W: np.ndarray) -> np.ndarray:
    m = np.size(y)

    y_pred = X @ weigths

    dp_l2 = X.T @ W @ (y_pred - y) / m
    return dp_l2

x0 = np.array([1.0, 1.0])
X = np.array([[1.0, 2.0], [1.0, 2.0], [1.0, 2.0]])
y = np.array([2.0, 2.0, 2.0])
weights = np.array([2.0, 2.0])
W = lc_weight(X, x0, np.float64(1))
lc_w_gradient(x0, X, y, weights, W)

In [ ]:
def lc_weight_gradient_descent(x0_input: np.ndarray, X_input: np.ndarray, y:np.ndarray, bandwidth: np.float64, num_iter: int, lr: float):
    m = np.size(y)
    ones = np.ones((m, 1))
    X = np.hstack((ones, X_input))
    x0 = np.hstack((np.ones(1), x0_input))
    W = lc_weight(X, x0, bandwidth)

    np.random.seed(42)
    weigths = np.random.rand(X.shape[1])

    print(f"First Error: {loss_fn(X, y, weigths):.10f}")

    for i in range(num_iter):
        weigths -= lr*lc_w_gradient(x0, X, y, weigths, W)
        print(f"Current ({i}) Error:  {loss_fn(X, y, weigths):.10f}", end='\r')
    print()
    return weigths

X = np.array([[2], [2], [2]])
y = np.array([3, 3, 3])
x0 = np.array([1.0])
ITER = 1000
LR = 0.01
BW = np.float64(2.0)

lc_weight_gradient_descent(x0, X, y, BW, ITER, LR)

In [ ]:
from sklearn.preprocessing import StandardScaler
csv_path = os.path.join(os.path.join("datasets", "housing"), "housing.csv")
df = pd.read_csv(csv_path, index_col=False)
df["ocean_proximity"] = 0.0
df = df.fillna(df.median())  # Fill NaN with column medians
df["rooms_per_house"] = df["total_rooms"] / df["households"]

TARGET_COL = 'median_house_value'
FEATURES_COL = ["housing_median_age", "rooms_per_house", "median_income"]

# Standardize X and y
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X = scaler_X.fit_transform(df[FEATURES_COL].to_numpy())
y = scaler_y.fit_transform(df[TARGET_COL].to_numpy().reshape(-1, 1)).flatten()
x0 = X[0]

ITER = 10000
LR = 0.001
BW = np.float64(1.0)

lc_weight_gradient_descent(x0, X, y, BW, ITER, LR)

# Regressão Logística

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    # Avoiding overflow issues with large values of z
    # z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

In [ ]:
def accuracy(y: np.ndarray, y_pred_probs: np.ndarray, threshold=0.5) -> np.float64:
    y_test = np.asarray(y).astype(int)
    y_pred_probs = np.asarray(y_pred_probs)
    y_pred = (y_pred_probs > threshold).astype(int)
    acc = (y_pred == y_test).mean()*100

    return acc

def bin_class_gradient(X: np.ndarray, y: np.ndarray, weights: np.ndarray) -> np.ndarray:
    p = sigmoid(X @ weights)
    grad = X.T @ (p - y)
    return grad

In [ ]:
def fit_bin_classification(X_input: np.ndarray, y:np.ndarray, num_iter: int, lr: float):
    m = np.size(y)
    ones = np.ones((m, 1))
    X = np.hstack((ones, X_input))
    weights = np.zeros(X.shape[1])

    print(f"First Acc: {accuracy(y, X @ weights)}%")

    for i in range(num_iter):
        weights -= lr*(bin_class_gradient(X, y, weights) / m)
        print(f"Acc {i}: {accuracy(y, X @ weights)}%", end="\r")
    print()
    return weights

X = np.array([[3], [1], [2]])
y = np.array([1, 0, 1])
ITER = 1000
LR = 0.1

weights = fit_bin_classification(X, y, ITER, LR)

X_teste = np.array([[1, 3], [1, 1], [1, 2]])

z = X_teste @ weights
preds = sigmoid(z)
print(preds);

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

TARGET_COL = 'median_house_value'
FEATURES_COL = ["housing_median_age", "rooms_per_house", "median_income"]

csv_path = os.path.join(os.path.join("datasets", "housing"), "housing.csv")
df = pd.read_csv(csv_path, index_col=False)


df["ocean_proximity"] = 0.0
df = df.fillna(df.median())  # Fill NaN with column medians
df["rooms_per_house"] = df["total_rooms"] / df["households"]


df = df.fillna(df.median())  # Fill NaN with column medians
df["rooms_per_house"] = df["total_rooms"] / df["households"]

# Standardize X and y
scaler_X = StandardScaler()
X = scaler_X.fit_transform(df[FEATURES_COL].to_numpy())

threshold = df[TARGET_COL].median()
y = (df[TARGET_COL].values > threshold).astype(int)


ITER = 10000
LR = 0.01

fit_bin_classification(X, y, ITER, LR)

# m = np.size(y)
# ones = np.ones((m, 1))
# X_test = np.hstack((ones, X))

# custom_preds = X_test @ weights

# print("Custom Result:", mse(y, custom_preds))

comparison_model = LogisticRegression()
comparison_model.fit(X, y)

y_pred = comparison_model.predict(X)

print(f"Lib Result: {accuracy(y, y_pred):.10f}");